In [ ]:
import torch
import torch.nn as nn
import torch.linalg as la
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import requests
import tarfile
from PIL import Image
import sys
import hydra
from omegaconf import OmegaConf

# Add the project source directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.models.dnf import DNFNetwork
from src.data.dataset import load_mnist
from src.utils.losses import compute_logits
from src.train import get_target_distributions
from src.utils.evaluation import (
    get_all_predictions,
    get_classification_report_and_cm,
    calculate_ece_and_reliability_diagram,
    calculate_nll_and_brier_score,
    get_ood_confidences_and_plot,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
# Define the configuration structure and load the files in a Hydra-like way
conf_path = os.path.join(project_root, 'conf')

# Start with the base config, which may contain defaults or other top-level settings
cfg = OmegaConf.load(os.path.join(conf_path, 'config.yaml'))

# Define which specific configs to load for this run
# This makes it easy to switch models or datasets later
model_config_name = "dnf_resnet.yaml" 
data_config_name = "mnist.yaml"
training_config_name = "default.yaml"

# Create a structured config by merging the components
cfg = OmegaConf.merge(
    cfg,
    {
        "model": OmegaConf.load(os.path.join(conf_path, 'model', model_config_name)),
        "data": OmegaConf.load(os.path.join(conf_path, 'data', data_config_name)),
        "training": OmegaConf.load(os.path.join(conf_path, 'training', training_config_name)),
    }
)

In [ ]:
# ----------------------------
# Data Loading
# ----------------------------
# The load_mnist function from the project returns DataLoaders.
# We need the underlying dataset for some evaluation metrics.
cfg.data.dataset.path = os.path.join(project_root, cfg.data.dataset.path)
_, test_loader = load_mnist(cfg.data)
test_dataset = test_loader.dataset

print(f"Loaded MNIST test dataset with {len(test_dataset)} samples.")

In [ ]:
# --- 1. Instantiate a new model ---
model = hydra.utils.instantiate(cfg.model, _convert_="partial").to(device)

# --- 2. Load the checkpoint file ---
CHECKPOINT_PATH = "" # Specify
if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Checkpoint file not found at '{CHECKPOINT_PATH}'. Please update the path.")

print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)

# --- 3. Load the states into the new instances ---
model.load_state_dict(checkpoint['model_state_dict'])

# Load the trained means and variances for correct evaluation
trainable_means = checkpoint['trainable_means'].to(device)
trainable_log_vars = checkpoint['trainable_log_vars'].to(device)

# --- 4. Set the model to evaluation mode ---
model.eval()

print("Model loaded successfully and set to evaluation mode.")

In [ ]:
# --- 1. Select a single image from the test set ---
image_idx = 6  # You can change this index to test different images
image, label = test_dataset[image_idx]
image = image.unsqueeze(0).to(device) # Add batch dimension and send to device

# --- 2. Perform inference (forward pass) ---
with torch.no_grad():
    # Get the final latent vector and log determinant
    outputs = model(image)
    z_final, total_log_det = outputs[-1]
    
    # Compute the logits (log-likelihoods) for each class
    final_target_dists = get_target_distributions(trainable_means, trainable_log_vars, cfg.training.num_classes)
    logits = compute_logits(z_final, total_log_det, final_target_dists)
    
    # Get the predicted class and confidence
    probabilities = torch.softmax(logits, dim=1)
    confidence, predicted_class = torch.max(probabilities, dim=1)

# --- 3. Generate a new image (inverse pass) ---
with torch.no_grad():
    # Sample a point from the latent distribution of the *predicted* class
    z_sample = final_target_dists[predicted_class.item()].sample()
    
    # The DNF model expects a 4D tensor for the inverse pass
    # We need to reshape the z_sample accordingly
    # For DNF, the latent space is flat, so we reshape to (1, C*H*W)
    # and then to the 4D shape the model's inverse method expects.
    # Assuming the latent space has the same shape as the input image's flattened version.
    z_sample_flat = z_sample.view(1, -1)
    
    # The inverse function needs to know the shape to start with.
    # For a simple DNF, it's the shape after the first squeeze.
    c, h, w = cfg.model.in_channels * 4, 28 // 2, 28 // 2
    z_sample_4d = z_sample_flat.view(1, c, h, w)
    
    generated_image, _ = model.inverse(z_sample_4d)

# --- 4. Visualize the results ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Original Image
axes[0].imshow(image.squeeze().cpu().numpy(), cmap='gray')
axes[0].set_title(f"Original Image\nTrue Label: {label}\nPredicted: {predicted_class.item()} (Conf: {confidence.item():.2f})")
axes[0].axis('off')

# Generated Image
axes[1].imshow(generated_image.squeeze().cpu().numpy(), cmap='gray')
axes[1].set_title(f"Generated Image\nFrom Class {predicted_class.item()} Latent Space")
axes[1].axis('off')

plt.show()